# 01_07_build_amortization_drp

Тетрадка для полного пересчета амортизации по терминалам за период Jan-Jul 2026
(граница периода: `2026-08-01`) и полной перезаписи таблицы
`sandbox_ai.shestopalov_terminal_amortization_model`.

Пайплайн:
1. Выгрузка активных терминалов по месяцам периода из Lake.
2. Выгрузка моделей терминалов из CDWH (`BMRT.BM_DET_DEVICE`) батчами.
3. Join с ценами моделей из `term_model.xlsx`.
4. Расчет `amortization_monthly = price / 48`, окна 48 месяцев и `amortization_for_report_month`.
5. Полная перезагрузка целевой таблицы в Datalake/Impala (`DROP + CREATE + ORC load`).


In [ ]:
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Параметры периода: Jan-Jul 2026 (граница периода: 2026-08-01)
period_start = '2026-01-01'
period_end_exclusive = '2026-08-01'
period_months = pd.date_range(period_start, pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1), freq='MS')

term_model_excel_path = '/home/jovyan/documents/Equaring/Data/term_model.xlsx'
source_file = 'term_model.xlsx'

target_table = 'sandbox_ai.shestopalov_terminal_amortization_model'
save_table_orc_name = 'shestopalov_terminal_amortization_model_2026_01_2026_06.orc'

# CDWH credentials (as agreed earlier)
khd_user = 'DS_LII'
khd_password = 'dl3S$wolx5dz'

print('period_months =', [m.strftime('%Y-%m') for m in period_months])
print('term_model_excel_path =', term_model_excel_path)
print('target_table =', target_table)


def clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in {'None', 'nan', 'NaN'}:
            out.append(s)
    return sorted(set(out))


def sql_in(values):
    vals = clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def norm_model(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().lower().replace('\xa0', ' ')
    if s in {'none', 'nan', 'null', '<null>', 'nat'}:
        return None
    s = s.replace('ё', 'е')
    s = re.sub(r'\s+', ' ', s)
    return s if s else None


def norm_terminal_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s.upper()




def clean_cdwh_text(v):
    """Normalize CDWH text: real nulls stay NaN (never the string 'None'/'nan')."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() in {"none", "nan", "null", "<null>", "nat"}:
        return np.nan
    return s


def is_blank_series(s):
    """True where value is null / empty / literal None|nan|null."""
    def _blank(v):
        if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
            return True
        t = str(v).strip()
        return t == "" or t.lower() in {"none", "nan", "null", "<null>", "nat"}

    return s.map(_blank)

def iter_chunks(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def split_nter_for_id_and_code(values):
    id_numeric = []
    code_values = []
    for v in values:
        s = str(v).strip()
        if not s:
            continue
        code_values.append(s)
        if re.fullmatch(r'\d+', s):
            id_numeric.append(str(int(s)))
    return sorted(set(id_numeric)), sorted(set(code_values))


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

dl = connect(
    to='DATALAKE',
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
dl._init_connection()

cdwh_connection = connect(
    to='CDWH',
    user_params={
        'user_name': khd_user,
        'password': khd_password,
    }
)
cdwh_connection._init_connection()

print('Impala + Datalake + CDWH initialized')

In [ ]:
# 1) Активные терминалы по месяцам периода + first_d_ter_delivery по serial
month_rows_sql = []
for m in period_months:
    month_start = m.strftime('%Y-%m-%d')
    month_end = (m + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    month_rows_sql.append(
        f"select cast('{month_start}' as date) as snapshot_month_start, cast('{month_end}' as date) as snapshot_month_end"
    )
months_sql = '\nunion all\n'.join(month_rows_sql)

sql_terminal_months = f"""
with months as (
{months_sql}
),
base as (
    select
      cast(t.c_nter as string) as c_nter,
      cast(t.c_pos_serial as string) as c_pos_serial,
      cast(t.d_ter_install as date) as d_ter_install,
      cast(t.d_ter_close as date) as d_ter_close,
      cast(t.d_ter_delivery as date) as d_ter_delivery
    from ods_alpha.scd1_pos_terminals t
    where t.c_nter is not null
      and t.c_pos_serial is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
      and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) < cast('{period_end_exclusive}' as date)
      and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{period_start}' as date)
),
first_deliver as (
    select
      cast(t.c_pos_serial as string) as c_pos_serial,
      min(cast(t.d_ter_delivery as date)) as first_d_ter_delivery
    from ods_alpha.scd1_pos_terminals t
    where t.c_pos_serial is not null
      and t.d_ter_delivery is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
    group by cast(t.c_pos_serial as string)
),
active_raw as (
    select
      cast(m.snapshot_month_start as date) as snapshot_month_start,
      cast(b.c_nter as string) as c_nter,
      cast(b.c_pos_serial as string) as c_pos_serial,
      cast(fd.first_d_ter_delivery as date) as first_d_ter_delivery,
      cast(b.d_ter_install as date) as d_ter_install,
      cast(b.d_ter_close as date) as d_ter_close,
      row_number() over (
        partition by cast(m.snapshot_month_start as date), cast(b.c_nter as string)
        order by
          coalesce(cast(b.d_ter_install as date), cast('1900-01-01' as date)) desc,
          coalesce(cast(b.d_ter_close as date), cast('2999-12-31' as date)) desc,
          cast(b.c_pos_serial as string) desc
      ) as rn
    from months m
    join base b
      on coalesce(cast(b.d_ter_install as date), cast('1900-01-01' as date)) <= m.snapshot_month_end
     and coalesce(cast(b.d_ter_close as date), cast('2999-12-31' as date)) >= m.snapshot_month_start
    left join first_deliver fd
      on fd.c_pos_serial = b.c_pos_serial
)
select
  cast(snapshot_month_start as string) as snapshot_month_start,
  cast(c_nter as string) as c_nter,
  cast(c_pos_serial as string) as c_pos_serial,
  cast(first_d_ter_delivery as date) as first_d_ter_delivery,
  cast(d_ter_install as date) as d_ter_install,
  cast(d_ter_close as date) as d_ter_close
from active_raw
where rn = 1
"""

with imp:
    imp.execute('set MEM_LIMIT=16g')
    terminals_monthly_df = imp.fetch(sql_terminal_months)

if terminals_monthly_df is None:
    terminals_monthly_df = pd.DataFrame(
        columns=['snapshot_month_start', 'c_nter', 'c_pos_serial', 'first_d_ter_delivery', 'd_ter_install', 'd_ter_close']
    )

for c in ['snapshot_month_start', 'c_nter', 'c_pos_serial']:
    if c in terminals_monthly_df.columns:
        terminals_monthly_df[c] = terminals_monthly_df[c].astype(str).str.strip()

terminals_monthly_df['nter_norm'] = terminals_monthly_df['c_nter'].apply(norm_terminal_key)

print('terminals_monthly rows =', len(terminals_monthly_df))
print('distinct c_nter =', terminals_monthly_df['c_nter'].nunique() if len(terminals_monthly_df) else 0)
display(terminals_monthly_df.head(10))

In [ ]:
# 2) Модели терминалов из CDWH (Oracle-safe batching)
nter_values = clean_keys(terminals_monthly_df['c_nter'].tolist()) if len(terminals_monthly_df) else []
id_values_numeric, code_values = split_nter_for_id_and_code(nter_values)

cdwh_id_batches = 0
cdwh_code_batches = 0
cdwh_code_fallback_batches = 0
bm_chunks = []

if nter_values:
    with cdwh_connection:
        if id_values_numeric:
            for id_chunk in iter_chunks(id_values_numeric, 950):
                id_in = ', '.join(id_chunk)
                sql_by_id = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.ID_DEVICE in ({id_in})
                """
                part_df = cdwh_connection.fetch(sql_by_id)
                cdwh_id_batches += 1
                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

        if code_values:
            for code_chunk in iter_chunks(code_values, 900):
                code_in = sql_in(code_chunk)
                sql_by_code_fast = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.CODE_DEVICE in ({code_in})
                """
                try:
                    part_df = cdwh_connection.fetch(sql_by_code_fast)
                    cdwh_code_batches += 1
                except Exception as exc:
                    sql_by_code_fallback = f"""
                    select distinct
                      trim(to_char(d.ID_DEVICE)) as device,
                      trim(to_char(d.CODE_DEVICE)) as code_device,
                      trim(to_char(d.MODEL_DEVICE)) as model_device
                    from BMRT.BM_DET_DEVICE d
                    where trim(to_char(d.CODE_DEVICE)) in ({code_in})
                    """
                    part_df = cdwh_connection.fetch(sql_by_code_fallback)
                    cdwh_code_fallback_batches += 1
                    print(f'CODE_DEVICE fallback chunk due to: {type(exc).__name__}')

                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

if bm_chunks:
    bm_det_device_df = pd.concat(bm_chunks, ignore_index=True)
else:
    bm_det_device_df = pd.DataFrame(columns=['device', 'code_device', 'model_device'])

bm_det_device_df.columns = [str(c).strip().lower() for c in bm_det_device_df.columns]
for col in ['device', 'code_device', 'model_device']:
    if col not in bm_det_device_df.columns:
        bm_det_device_df[col] = np.nan
    bm_det_device_df[col] = bm_det_device_df[col].map(clean_cdwh_text)

bm_det_device_df['device_norm'] = bm_det_device_df['device'].apply(norm_terminal_key)
bm_det_device_df['code_device_norm'] = bm_det_device_df['code_device'].apply(norm_terminal_key)
_before_bm = len(bm_det_device_df)
bm_det_device_df = bm_det_device_df.loc[bm_det_device_df['model_device'].notna()].copy()
print(f'bm_det_device: dropped empty MODEL_DEVICE rows: {_before_bm - len(bm_det_device_df):,}')
bm_det_device_df = bm_det_device_df.drop_duplicates(subset=['device', 'code_device', 'model_device'])

print('CDWH batch stats:')
print('  by_id =', cdwh_id_batches)
print('  by_code_fast =', cdwh_code_batches)
print('  by_code_fallback =', cdwh_code_fallback_batches)
print('bm_det_device rows =', len(bm_det_device_df))
display(bm_det_device_df.head(10))


In [ ]:
# 3) Цены моделей из Excel + join к terminal-month perimeter
price_src_df = pd.read_excel(term_model_excel_path)
required_price_cols = ['model_name', 'price']
missing_price_cols = [c for c in required_price_cols if c not in price_src_df.columns]
if missing_price_cols:
    raise RuntimeError(f'В term_model.xlsx отсутствуют колонки: {missing_price_cols}')

price_map_df = price_src_df[['model_name', 'price']].copy()
price_map_df['model_key'] = price_map_df['model_name'].apply(norm_model)
price_map_df['price'] = pd.to_numeric(price_map_df['price'], errors='coerce')
price_map_df = (
    price_map_df.dropna(subset=['model_key', 'price'])
    .groupby('model_key', as_index=False)
    .agg(price=('price', 'max'))
)

# Model mapping priority: ID_DEVICE first, then CODE_DEVICE
if len(bm_det_device_df):
    device_map_df = (
        bm_det_device_df.loc[
            bm_det_device_df['device_norm'].notna()
            & bm_det_device_df['model_device'].notna(),
            ['device_norm', 'model_device']
        ]
        .drop_duplicates(subset=['device_norm'], keep='first')
    )
    code_map_df = (
        bm_det_device_df.loc[
            bm_det_device_df['code_device_norm'].notna()
            & bm_det_device_df['model_device'].notna(),
            ['code_device_norm', 'model_device']
        ]
        .drop_duplicates(subset=['code_device_norm'], keep='first')
    )
    device_map = dict(zip(device_map_df['device_norm'], device_map_df['model_device']))
    code_map = dict(zip(code_map_df['code_device_norm'], code_map_df['model_device']))
else:
    device_map = {}
    code_map = {}

amort_base_df = terminals_monthly_df.copy()
amort_base_df['model_device'] = amort_base_df['nter_norm'].map(device_map)
amort_base_df['join_key_used'] = np.where(
    amort_base_df['model_device'].notna(), 'id_device', None
)

need_code_mask = amort_base_df['model_device'].isna()
amort_base_df.loc[need_code_mask, 'model_device'] = amort_base_df.loc[need_code_mask, 'nter_norm'].map(code_map)
amort_base_df.loc[need_code_mask & amort_base_df['model_device'].notna(), 'join_key_used'] = 'code_device'

# sanitize again (never keep literal 'None' / 'nan')
amort_base_df['model_device'] = amort_base_df['model_device'].map(clean_cdwh_text)
amort_base_df.loc[amort_base_df['model_device'].isna(), 'join_key_used'] = None

amort_base_df['model_key'] = amort_base_df['model_device'].apply(norm_model)
amort_base_df.loc[amort_base_df['model_key'].isna(), ['model_device', 'join_key_used']] = np.nan

amort_base_df = amort_base_df.merge(price_map_df, on='model_key', how='left')

print('price_map rows =', len(price_map_df))
print('amort_base rows =', len(amort_base_df))
print(
    'model found =', int(amort_base_df['model_device'].notna().sum()),
    '| price found =', int(pd.to_numeric(amort_base_df['price'], errors='coerce').notna().sum()),
)
display(amort_base_df.head(10))


In [ ]:
# 4) Расчет амортизации по месяцам
amort_df = amort_base_df.copy()

amort_df['snapshot_month_start'] = pd.to_datetime(amort_df['snapshot_month_start'], errors='coerce')
amort_df['first_d_ter_delivery'] = pd.to_datetime(amort_df['first_d_ter_delivery'], errors='coerce')

amort_df['missing_serial'] = is_blank_series(amort_df['c_pos_serial'])
amort_df['missing_deliver'] = amort_df['first_d_ter_delivery'].isna()
# missing_model: no usable CDWH model (null / literal None / bad model_key)
amort_df['missing_model'] = (
    is_blank_series(amort_df['model_device'])
    | amort_df['model_key'].isna()
)
# missing_price: model known, but no Excel price (true CDWH→Excel gap)
amort_df['missing_price'] = (~amort_df['missing_model']) & pd.to_numeric(amort_df['price'], errors='coerce').isna()


def qc_status_fn(row):
    if row['missing_serial']:
        return 'missing_serial'
    if row['missing_deliver']:
        return 'missing_deliver'
    if row['missing_model']:
        return 'missing_model'
    if row['missing_price']:
        return 'missing_price'
    return 'complete'


amort_df['qc_status'] = amort_df.apply(qc_status_fn, axis=1)

complete_mask = amort_df['qc_status'] == 'complete'

first_month_first_day = amort_df.loc[complete_mask, 'first_d_ter_delivery'].dt.to_period('M').dt.to_timestamp()
report_month_first_day = amort_df.loc[complete_mask, 'snapshot_month_start'].dt.to_period('M').dt.to_timestamp()

months_from_start = (
    (report_month_first_day.dt.year - first_month_first_day.dt.year) * 12
    + (report_month_first_day.dt.month - first_month_first_day.dt.month)
)

amort_df['months_from_start'] = np.nan
amort_df.loc[complete_mask, 'months_from_start'] = months_from_start

amort_df['is_in_48m_window'] = (
    amort_df['months_from_start'].notna()
    & (amort_df['months_from_start'] >= 0)
    & (amort_df['months_from_start'] < 48)
)

amort_df['amortization_monthly'] = pd.to_numeric(amort_df['price'], errors='coerce') / 48.0
amort_df['amortization_for_report_month'] = amort_df['amortization_monthly'].where(amort_df['is_in_48m_window'], 0.0)
amort_df['amortization_for_report_month'] = amort_df['amortization_for_report_month'].fillna(0.0)

amort_df['report_month'] = amort_df['snapshot_month_start'].dt.strftime('%Y-%m')
amort_df['snapshot_month_start'] = amort_df['snapshot_month_start'].dt.strftime('%Y-%m-%d')
amort_df['first_d_ter_delivery'] = amort_df['first_d_ter_delivery'].dt.strftime('%Y-%m-%d')

amort_df['is_in_48m_window'] = amort_df['is_in_48m_window'].astype(int)
amort_df['is_amortized'] = (amort_df['amortization_for_report_month'] > 0).astype(int)
amort_df['load_dt'] = pd.Timestamp.now().strftime('%Y-%m-%d')
amort_df['source_file'] = source_file

qc_missing_df = (
    amort_df.groupby('qc_status', as_index=False)
    .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
    .sort_values(['rows', 'terminals_nunique'], ascending=False)
    .reset_index(drop=True)
)

join_cov_df = amort_df.copy()
join_cov_df['join_key_used'] = join_cov_df['join_key_used'].fillna('no_match')
join_coverage_df = (
    join_cov_df.groupby('join_key_used', as_index=False)
    .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
    .sort_values(['rows', 'terminals_nunique'], ascending=False)
    .reset_index(drop=True)
)

print('QC: missing reasons')
display(qc_missing_df)
print('QC: join coverage')
display(join_coverage_df)
print('amort_df rows =', len(amort_df))
print('amortized rows =', int((amort_df['is_amortized'] == 1).sum()))
display(amort_df.head(10))


## QC: модели из CDWH без цены в term_model.xlsx + складской c_nter

1. **Top-30 моделей CDWH без цены** в `term_model.xlsx`.
2. **Складской `c_nter`**: в сыром lake один номер на много serial (ожидаемо `10000051`); в `amort_df` после `rn=1` остаётся 1 serial на месяц — сравнение RAW vs post-collapse.


In [ ]:
# QC A) Top models present in CDWH mapping but missing from term_model.xlsx (no price)
has_model_no_price = amort_df.loc[
    (~amort_df['missing_model']) & amort_df['missing_price']
].copy()

top_missing_price_models = (
    has_model_no_price.groupby(['model_device', 'model_key'], as_index=False)
    .agg(
        rows=('c_nter', 'size'),
        terminals_nunique=('c_nter', 'nunique'),
        serials_nunique=('c_pos_serial', 'nunique'),
    )
    .sort_values(['terminals_nunique', 'rows'], ascending=False)
    .head(30)
    .reset_index(drop=True)
)

print('=== Top-30 models in CDWH but NOT in term_model.xlsx (missing price) ===')
if top_missing_price_models.empty:
    print('OK: no CDWH models without Excel price')
else:
    display(top_missing_price_models)

# QC B) Delivery exists, model missing (true gaps in CDWH / join)
delivery_no_model = amort_df.loc[
    (~amort_df['missing_deliver']) & amort_df['missing_model']
].copy()
top_nter_no_model = (
    delivery_no_model.groupby('c_nter', as_index=False)
    .agg(
        rows=('c_nter', 'size'),
        serials_nunique=('c_pos_serial', 'nunique'),
        sample_serial=('c_pos_serial', 'first'),
    )
    .sort_values(['serials_nunique', 'rows'], ascending=False)
    .head(30)
    .reset_index(drop=True)
)
print('=== Top-30 c_nter with delivery but NO model ===')
display(top_nter_no_model)
print(
    'rows with delivery & no model =',
    f'{len(delivery_no_model):,} / {len(amort_df):,}',
)

# QC C) Shared c_nter in CURRENT perimeter (after rn=1 collapse)
nter_serial_freq = (
    amort_df.groupby('c_nter', as_index=False)
    .agg(
        rows=('c_nter', 'size'),
        serials_nunique=('c_pos_serial', 'nunique'),
        months_nunique=('snapshot_month_start', 'nunique'),
    )
    .sort_values(['serials_nunique', 'rows'], ascending=False)
)
warehouse_like_nter = nter_serial_freq.loc[nter_serial_freq['serials_nunique'] >= 2].head(30)
print('=== Shared c_nter in amort_df (after month+c_nter rn=1 collapse), TOP-30 by serials ===')
print('NOTE: perimeter keeps only 1 serial per (month, c_nter), so warehouse id may look rare here.')
display(warehouse_like_nter)

probe_nter = '10000051'
probe_row = nter_serial_freq.loc[
    nter_serial_freq['c_nter'].map(norm_terminal_key).astype(str) == str(norm_terminal_key(probe_nter))
]
print(f'=== Frequency for c_nter={probe_nter} in amort_df (post-collapse) ===')
display(probe_row if len(probe_row) else pd.DataFrame([{'c_nter': probe_nter, 'note': 'not found in amort_df'}]))

# QC D) RAW lake: warehouse c_nter must have VERY many serials
sql_wh = (
    "select "
    "cast(c_nter as string) as c_nter, "
    "count(*) as rows_raw, "
    "count(distinct cast(c_pos_serial as string)) as serials_nunique "
    "from ods_alpha.scd1_pos_terminals "
    f"where cast(c_nter as string) = '{probe_nter}' "
    "and c_pos_serial is not null "
    "and coalesce(ods_deleted_flg, '0') <> '1' "
    "group by cast(c_nter as string)"
)
with imp:
    wh_raw_df = imp.fetch(sql_wh)
print(f'=== RAW lake serials for warehouse c_nter={probe_nter} ===')
display(
    wh_raw_df if wh_raw_df is not None and len(wh_raw_df)
    else pd.DataFrame([{'c_nter': probe_nter, 'note': 'not found in lake'}])
)

sql_top_shared = (
    "select "
    "cast(c_nter as string) as c_nter, "
    "count(distinct cast(c_pos_serial as string)) as serials_nunique, "
    "count(*) as rows_raw "
    "from ods_alpha.scd1_pos_terminals "
    "where c_nter is not null "
    "and c_pos_serial is not null "
    "and coalesce(ods_deleted_flg, '0') <> '1' "
    "group by cast(c_nter as string) "
    "having count(distinct cast(c_pos_serial as string)) >= 50 "
    "order by serials_nunique desc "
    "limit 30"
)
with imp:
    top_shared_raw = imp.fetch(sql_top_shared)
print('=== RAW lake TOP-30 shared c_nter (serials >= 50) ===')
display(top_shared_raw)


## Probe: конкретный терминал

`c_nter = 10000051`, `c_pos_serial = 2331719393`

Смотрим глазами все ступени: lake perimeter → CDWH model → Excel price → 48m window → amort amount.


In [ ]:
# Probe one terminal end-to-end
PROBE_NTER = '10000051'
PROBE_SERIAL = '2331719393'

probe_nter_norm = norm_terminal_key(PROBE_NTER)

print('=== 1) Rows in terminals_monthly_df ===')
tm = terminals_monthly_df.copy()
tm_hit = tm.loc[
    (tm['c_nter'].map(norm_terminal_key) == probe_nter_norm)
    | (tm['c_pos_serial'].astype(str).str.strip() == PROBE_SERIAL)
]
display(tm_hit.sort_values(['snapshot_month_start', 'c_pos_serial']).head(50))
print(
    'rows=', len(tm_hit),
    '| serials=', tm_hit['c_pos_serial'].nunique(),
    '| months=', tm_hit['snapshot_month_start'].nunique(),
)

print('=== 2) CDWH BM_DET_DEVICE hits for this c_nter ===')
bm_hit = bm_det_device_df.loc[
    (bm_det_device_df['device_norm'] == probe_nter_norm)
    | (bm_det_device_df['code_device_norm'] == probe_nter_norm)
    | (bm_det_device_df['device'].astype(str) == PROBE_NTER)
    | (bm_det_device_df['code_device'].astype(str) == PROBE_NTER)
]
display(bm_hit)
print(
    'in device_map =', probe_nter_norm in device_map,
    '| in code_map =', probe_nter_norm in code_map,
)
if probe_nter_norm in device_map:
    print('device_map model =', device_map[probe_nter_norm])
if probe_nter_norm in code_map:
    print('code_map model =', code_map[probe_nter_norm])

print('=== 3) amort_df rows for nter/serial ===')
ad = amort_df.copy()
ad_hit = ad.loc[
    (ad['c_nter'].map(norm_terminal_key) == probe_nter_norm)
    | (ad['c_pos_serial'].astype(str).str.strip() == PROBE_SERIAL)
]
cols_show = [
    c for c in [
        'snapshot_month_start', 'report_month', 'c_nter', 'c_pos_serial',
        'first_d_ter_delivery', 'model_device', 'model_key', 'join_key_used',
        'price', 'amortization_monthly', 'months_from_start', 'is_in_48m_window',
        'amortization_for_report_month', 'qc_status',
        'missing_serial', 'missing_deliver', 'missing_model', 'missing_price',
    ] if c in ad_hit.columns
]
display(ad_hit[cols_show].sort_values(['snapshot_month_start', 'c_pos_serial']))

print('=== 4) How many serials share this c_nter in amort_df? ===')
share = ad.loc[ad['c_nter'].map(norm_terminal_key) == probe_nter_norm]
print(
    'rows=', len(share),
    '| unique serials=', share['c_pos_serial'].nunique(),
    '| unique models=', share['model_device'].nunique(dropna=True),
)
if len(share):
    display(
        share.groupby(['model_device', 'model_key', 'qc_status'], dropna=False, as_index=False)
        .agg(rows=('c_pos_serial', 'size'), serials=('c_pos_serial', 'nunique'))
        .sort_values('serials', ascending=False)
        .head(20)
    )

print('=== 5) Exact serial probe ===')
serial_hit = ad.loc[ad['c_pos_serial'].astype(str).str.strip() == PROBE_SERIAL, cols_show]
display(serial_hit.sort_values('snapshot_month_start'))
if serial_hit.empty:
    print('WARNING: serial not found in amort_df — check lake perimeter filters / ods_deleted / date window')
else:
    print('qc_status value counts:')
    display(serial_hit['qc_status'].value_counts(dropna=False))


In [ ]:
# 5) Итоговый датафрейм для загрузки в target table
load_cols = [
    'snapshot_month_start',
    'report_month',
    'c_nter',
    'c_pos_serial',
    'first_d_ter_delivery',
    'model_device',
    'model_key',
    'price',
    'amortization_monthly',
    'months_from_start',
    'is_in_48m_window',
    'amortization_for_report_month',
    'is_amortized',
    'join_key_used',
    'qc_status',
    'missing_serial',
    'missing_deliver',
    'missing_model',
    'missing_price',
    'load_dt',
    'source_file',
]

for c in load_cols:
    if c not in amort_df.columns:
        amort_df[c] = None

final_load_df = amort_df[load_cols].copy()

# Однозначность ключа внутри периода
dup_check = (
    final_load_df.groupby(['snapshot_month_start', 'c_nter'], as_index=False)
    .size()
    .rename(columns={'size': 'cnt'})
)
dup_cnt = int((dup_check['cnt'] > 1).sum()) if len(dup_check) else 0
print('duplicate keys (snapshot_month_start, c_nter) =', dup_cnt)
if dup_cnt:
    display(dup_check[dup_check['cnt'] > 1].head(20))
    raise RuntimeError('Найдены дубли по ключу (snapshot_month_start, c_nter). Загрузка остановлена.')

print('final_load_df rows =', len(final_load_df))
print('final_load_df months =', sorted(final_load_df['report_month'].dropna().astype(str).unique().tolist()))
display(final_load_df.head(20))

In [ ]:
# 6) Полная перезапись target table в Datalake/Impala (DROP + CREATE + ORC load)

def load_to_datalake_from_file(local_file_path: str, table: str, dest_catalog: str = 'external', cleanup_before_copy: bool = True):
    schema = table.split('.')[0]
    table_name = table.split('.')[-1]
    hdfs_path = f'/warehouse/tablespace/{dest_catalog}/hive/{schema}.db/{table_name}'

    if cleanup_before_copy:
        # Удаляем старые файлы, чтобы избежать дублей после полной перезагрузки
        subprocess.run(['hdfs', 'dfs', '-rm', '-r', '-f', f'{hdfs_path}/*'], check=False)

    subprocess.run(['hdfs', 'dfs', '-copyFromLocal', '-f', local_file_path, f'{hdfs_path}/{local_file_path}'], check=True)
    out = subprocess.run(['hdfs', 'dfs', '-ls', '-h', hdfs_path], capture_output=True, text=True, check=True)
    print(f'Files in HDFS path {hdfs_path}:\n{out.stdout}')


# Подготовка типов и null для записи
load_df = final_load_df.copy()
for c in ['price', 'amortization_monthly', 'months_from_start', 'amortization_for_report_month']:
    load_df[c] = pd.to_numeric(load_df[c], errors='coerce')
for c in ['is_in_48m_window', 'is_amortized', 'missing_serial', 'missing_deliver', 'missing_model', 'missing_price']:
    load_df[c] = pd.to_numeric(load_df[c], errors='coerce').fillna(0).astype('int64')

load_df = load_df.fillna({
    'snapshot_month_start': '',
    'report_month': '',
    'c_nter': '',
    'c_pos_serial': '',
    'first_d_ter_delivery': '',
    'model_device': '',
    'model_key': '',
    'join_key_used': '',
    'qc_status': '',
    'load_dt': '',
    'source_file': source_file,
})

load_df.to_orc(save_table_orc_name, index=False)
print('ORC prepared:', save_table_orc_name, 'rows=', len(load_df))

create_sql = f"""
create external table if not exists {target_table} (
    snapshot_month_start string,
    report_month string,
    c_nter string,
    c_pos_serial string,
    first_d_ter_delivery string,
    model_device string,
    model_key string,
    price double,
    amortization_monthly double,
    months_from_start double,
    is_in_48m_window bigint,
    amortization_for_report_month double,
    is_amortized bigint,
    join_key_used string,
    qc_status string,
    missing_serial bigint,
    missing_deliver bigint,
    missing_model bigint,
    missing_price bigint,
    load_dt string,
    source_file string
)
stored as orc
 tblproperties ('transactional'='false')
"""

with dl:
    dl.execute(f'drop table if exists {target_table}')
    dl.execute(create_sql)

load_to_datalake_from_file(save_table_orc_name, target_table, dest_catalog='external', cleanup_before_copy=True)

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')

print('Reload completed:', target_table)

In [ ]:
# 7) Post-load DQ checks
sql_dq = f"""
with base as (
    select *
    from {target_table}
),
dup as (
    select snapshot_month_start, c_nter, count(*) as cnt
    from base
    group by snapshot_month_start, c_nter
    having count(*) > 1
)
select
    (select count(*) from base) as rows_cnt,
    (select count(distinct c_nter) from base) as distinct_c_nter_cnt,
    (select count(distinct snapshot_month_start) from base) as months_cnt,
    (select count(*) from dup) as duplicated_month_nter_cnt,
    (select sum(case when qc_status = 'complete' then 1 else 0 end) from base) as complete_rows_cnt,
    (select sum(case when qc_status <> 'complete' then 1 else 0 end) from base) as non_complete_rows_cnt,
    (select sum(coalesce(amortization_for_report_month, 0.0)) from base) as amortization_total
"""

sql_qc_status = f"""
select
  qc_status,
  count(*) as rows_cnt,
  count(distinct c_nter) as terminals_nunique
from {target_table}
group by qc_status
order by rows_cnt desc, qc_status
"""

sql_monthly = f"""
select
  report_month,
  count(*) as rows_cnt,
  count(distinct c_nter) as terminals_nunique,
  sum(coalesce(amortization_for_report_month, 0.0)) as amortization_total
from {target_table}
group by report_month
order by report_month
"""

sql_sample = f"""
select *
from {target_table}
order by report_month, c_nter
limit 50
"""

with imp:
    dq_summary_df = imp.fetch(sql_dq)
    qc_status_df = imp.fetch(sql_qc_status)
    monthly_df = imp.fetch(sql_monthly)
    sample_df = imp.fetch(sql_sample)

print('DQ summary:')
display(dq_summary_df)
print('QC by status:')
display(qc_status_df)
print('Monthly control totals:')
display(monthly_df)
print('Sample rows:')
display(sample_df)

In [ ]:
# 7b) Smoke after reload: expect Jan–Jul including 2026-07
expected_months = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']

sql_monthly_smoke = f"""
select
  cast(report_month as string) as report_month,
  count(*) as rows_cnt,
  count(distinct cast(c_nter as string)) as terminals_nunique,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total
from {target_table}
group by cast(report_month as string)
order by 1
"""

sql_july_smoke = f"""
select
  count(*) as july_rows,
  count(distinct cast(c_nter as string)) as july_terminals,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as july_amort
from {target_table}
where cast(report_month as string) = '2026-07'
   or cast(snapshot_month_start as string) = '2026-07-01'
"""

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')
    monthly_smoke_df = imp.fetch(sql_monthly_smoke)
    july_smoke_df = imp.fetch(sql_july_smoke)

print('Smoke table:', target_table)
display(monthly_smoke_df)
display(july_smoke_df)

have = (
    set(monthly_smoke_df['report_month'].astype(str).str[:7].tolist())
    if monthly_smoke_df is not None and len(monthly_smoke_df)
    else set()
)
missing = [m for m in expected_months if m not in have]
print('months present:', sorted(have))
print('missing expected:', missing)

if missing:
    raise AssertionError(f'Missing months in amort model: {missing}')
if july_smoke_df is None or int(july_smoke_df.iloc[0]['july_rows'] or 0) <= 0:
    raise AssertionError('July 2026 is empty in amort model')

print('OK: July present, all expected months found')


## 7c) QC: объём амортизации (диагностика Excel ≪ lake)

Если lake ≫ Excel (×3–×6), проверяем:
1. долю терминалов в окне 48м (`is_in_48m_window`);
2. сумму amort / число списываемых терминалов по месяцам;
3. распределение `months_from_start`;
4. дубли ключа `(snapshot_month_start, c_nter)` в озере;
5. (опционально) один `c_nter` на нескольких `n_agr` в SA — двойной счёт в витрине.


In [ ]:
# 7c) QC: amortization volume diagnostics (post-load Impala + optional local amort_df)
from IPython.display import display

sql_vol_by_month = f"""
select
  cast(report_month as string) as report_month,
  count(*) as rows_cnt,
  sum(case when cast(is_in_48m_window as bigint) = 1 then 1 else 0 end) as in_window_cnt,
  sum(case when coalesce(cast(amortization_for_report_month as double), 0.0) > 0 then 1 else 0 end) as amort_positive_cnt,
  round(
    sum(case when cast(is_in_48m_window as bigint) = 1 then 1 else 0 end) / cast(count(*) as double),
    4
  ) as share_in_window,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total,
  avg(case
        when coalesce(cast(amortization_for_report_month as double), 0.0) > 0
        then cast(amortization_for_report_month as double)
      end) as avg_amort_when_positive,
  avg(case
        when cast(is_in_48m_window as bigint) = 1
        then cast(price as double)
      end) as avg_price_in_window
from {target_table}
group by cast(report_month as string)
order by 1
"""

sql_age_buckets = f"""
select
  cast(report_month as string) as report_month,
  case
    when months_from_start is null then 'null'
    when cast(months_from_start as double) < 0 then 'lt_0'
    when cast(months_from_start as double) < 12 then '0_11'
    when cast(months_from_start as double) < 24 then '12_23'
    when cast(months_from_start as double) < 36 then '24_35'
    when cast(months_from_start as double) < 48 then '36_47'
    else 'ge_48'
  end as age_bucket,
  count(*) as terminals_cnt,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total
from {target_table}
group by 1, 2
order by 1, 2
"""

sql_dup_keys = f"""
select snapshot_month_start, c_nter, count(*) as cnt
from {target_table}
group by snapshot_month_start, c_nter
having count(*) > 1
order by cnt desc
limit 50
"""

_sample_month = '2026-01-01'
sql_multi_agr = f"""
with amort_pos as (
  select distinct cast(c_nter as string) as c_nter
  from {target_table}
  where cast(snapshot_month_start as string) = '{_sample_month}'
    and coalesce(cast(amortization_for_report_month as double), 0.0) > 0
),
agr_map as (
  select
    cast(t.c_nter as string) as c_nter,
    cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_pos_terminals t
  join ods_alpha.scd1_agr_terms a
    on cast(a.c_nmrc as string) = cast(t.c_nmrc as string)
  join amort_pos p on p.c_nter = cast(t.c_nter as string)
  where coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and upper(trim(cast(a.cf_ter_type as string))) = 'P'
    and cast(a.d_valid_from as date) <= cast('{_sample_month}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) > cast('{_sample_month}' as date))
  group by cast(t.c_nter as string), cast(a.n_agr as string)
)
select
  count(*) as terminal_agr_pairs,
  count(distinct c_nter) as terminals,
  sum(case when agr_cnt > 1 then 1 else 0 end) as terminals_multi_agr
from (
  select c_nter, count(distinct n_agr) as agr_cnt
  from agr_map
  group by c_nter
) x
"""

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')
    vol_by_month_df = imp.fetch(sql_vol_by_month)
    age_buckets_df = imp.fetch(sql_age_buckets)
    dup_keys_df = imp.fetch(sql_dup_keys)
    try:
        multi_agr_df = imp.fetch(sql_multi_agr)
    except Exception as exc:
        print('multi-agr check skipped:', type(exc).__name__, str(exc)[:200])
        multi_agr_df = None

print('=== Amort volume by month ===')
display(vol_by_month_df)
print('=== Age buckets (months_from_start) ===')
display(age_buckets_df)
print('=== Duplicate (snapshot_month_start, c_nter) — expect 0 rows ===')
display(dup_keys_df)
if multi_agr_df is not None:
    print(f'=== Multi-agr risk for amort>0 terminals @ {_sample_month} ===')
    display(multi_agr_df)

if 'amort_df' in globals() and amort_df is not None and len(amort_df):
    _adf = amort_df.copy()
    _adf['_amort'] = pd.to_numeric(_adf.get('amortization_for_report_month'), errors='coerce').fillna(0)
    _adf['_win'] = pd.to_numeric(_adf.get('is_in_48m_window'), errors='coerce').fillna(0)
    loc = (
        _adf.assign(report_month=_adf['report_month'].astype(str))
        .groupby('report_month', as_index=False)
        .agg(
            rows=('c_nter', 'size'),
            in_window=('_win', 'sum'),
            amortization_total=('_amort', 'sum'),
        )
    )
    loc['share_in_window'] = loc['in_window'] / loc['rows']
    print('=== Local amort_df (before/alongside load) ===')
    display(loc)

if vol_by_month_df is not None and len(vol_by_month_df):
    _v = vol_by_month_df.copy()
    _v['share_in_window'] = pd.to_numeric(_v['share_in_window'], errors='coerce')
    high_share = _v.loc[_v['share_in_window'] >= 0.90, 'report_month'].astype(str).tolist()
    if high_share:
        print(
            'WARNING: share_in_window >= 90% for months:', high_share,
            '- почти весь парк в окне 48м; сверьте first_d_ter_delivery vs дату закупки Excel.'
        )
    if dup_keys_df is not None and len(dup_keys_df):
        print('WARNING: duplicate keys in amort model - витрина может раздувать SUM(amort).')
    if multi_agr_df is not None and len(multi_agr_df):
        _ma = int(pd.to_numeric(multi_agr_df.iloc[0].get('terminals_multi_agr'), errors='coerce') or 0)
        if _ma > 0:
            print(
                f'WARNING: {_ma} terminals with amort>0 map to multiple n_agr @ {_sample_month} - '
                'возможен двойной счёт в final_df при SUM по договорам.'
            )

_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')
qc_vol_csv = _qc_out_dir / 'amortization_volume_qc_by_month.csv'
if vol_by_month_df is not None:
    vol_by_month_df.to_csv(qc_vol_csv, index=False, encoding='utf-8-sig')
    print('Saved:', qc_vol_csv)


## 7d) QC: mid-month transfer serial в периметре `final_df`

Сколько физических устройств (`c_pos_serial`) в **нашем** периметре (ключи из `final_df`) в одном месяце жили у ≥2 `c_nter` / клиентов.

- Периметр: `final_df` (memory или CSV) → `agr_id`+`inn` → SA `n_agr` → `agr_terms` → `pos_terminals`.
- **handover_same_month**: ≥2 `c_nter` и ≥2 `n_cmp_client` (fallback: `n_agr`) на один serial в месяце.
- **close_then_install**: у одного `c_nter` close ∈ месяц, у другого install ∈ тот же месяц.
- Не путать с multi-agr (один `c_nter` на нескольких договорах) — это 7c.


In [ ]:
# 7d) QC: mid-month serial transfer within final_df perimeter
from IPython.display import display

FINAL_DF_CSV_CANDIDATES = [
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_07_mpos.csv'),
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_06_mpos.csv'),
]
AGR_CHUNK = 800
NAGR_CHUNK = 700
_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')


def _norm_agr_7d(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s in {'', 'nan', 'None', 'NaN'}:
        return None
    s = re.sub(r'\.0$', '', s)
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s


def _norm_inn_7d(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v).strip())
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    return s


def _load_final_df_for_7d():
    if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
        return final_df_period_df.copy(), 'memory:final_df_period_df'
    if 'final_df_by_month' in globals() and final_df_by_month:
        parts = []
        for m, g in final_df_by_month.items():
            if g is None or len(g) == 0:
                continue
            gg = g.copy()
            if 'report_month' not in gg.columns:
                gg['report_month'] = str(m)[:7]
            parts.append(gg)
        if parts:
            return pd.concat(parts, ignore_index=True), 'memory:final_df_by_month'
    for pth in FINAL_DF_CSV_CANDIDATES:
        if pth.exists():
            return pd.read_csv(pth, dtype=str, low_memory=False), f'csv:{pth}'
    raise RuntimeError(
        'final_df not found: set final_df_period_df / final_df_by_month in kernel '
        'or place final_df_period_2026_01_2026_07_mpos.csv under Equaring/Data'
    )


fdf_7d, fdf_7d_src = _load_final_df_for_7d()
print('final_df source:', fdf_7d_src, '| rows=', len(fdf_7d))

need_cols = {'report_month', 'inn', 'agr_id'}
missing_cols = need_cols - set(fdf_7d.columns)
if missing_cols:
    raise RuntimeError(f'final_df missing columns: {sorted(missing_cols)}')

keys_7d = fdf_7d[['report_month', 'inn', 'agr_id']].copy()
keys_7d['report_month'] = keys_7d['report_month'].astype(str).str.strip().str[:7]
keys_7d['inn_key'] = keys_7d['inn'].map(_norm_inn_7d)
keys_7d['agr_id_key'] = keys_7d['agr_id'].map(_norm_agr_7d)
keys_7d = keys_7d.dropna(subset=['report_month', 'inn_key', 'agr_id_key']).drop_duplicates()
print('unique (month, inn, agr_id) keys =', len(keys_7d))
print('months =', sorted(keys_7d['report_month'].unique().tolist()))

agr_ids_all = clean_keys(keys_7d['agr_id_key'].tolist())
agr_map_parts = []
with imp:
    imp.execute('set MEM_LIMIT=16g')
    for chunk in iter_chunks(agr_ids_all, AGR_CHUNK):
        agr_in = sql_in(chunk)
        sql_agr_map = f"""
        select distinct
          cast(a.abs_agr_id as string) as agr_id,
          cast(a.n_agr as string) as n_agr,
          cast(a.n_cmp_client as string) as n_cmp_client,
          cast(c.c_inn as string) as inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where cast(a.abs_agr_id as string) in ({agr_in})
          and upper(trim(cast(a.acq_class as string))) = 'SA'
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
        """
        part = imp.fetch(sql_agr_map)
        if part is not None and len(part):
            agr_map_parts.append(part)

if not agr_map_parts:
    raise RuntimeError('No SA agr_id → n_agr mapping for final_df keys')

agr_map_7d = pd.concat(agr_map_parts, ignore_index=True)
agr_map_7d['agr_id_key'] = agr_map_7d['agr_id'].map(_norm_agr_7d)
agr_map_7d['inn_key'] = agr_map_7d['inn'].map(_norm_inn_7d)
agr_map_7d['n_agr'] = agr_map_7d['n_agr'].astype(str).str.strip()
agr_map_7d['n_cmp_client'] = agr_map_7d['n_cmp_client'].astype(str).str.strip()
agr_map_7d = agr_map_7d.dropna(subset=['agr_id_key', 'inn_key', 'n_agr']).drop_duplicates(
    subset=['agr_id_key', 'inn_key', 'n_agr']
)

keys_nagr = keys_7d.merge(
    agr_map_7d[['agr_id_key', 'inn_key', 'n_agr', 'n_cmp_client']],
    on=['agr_id_key', 'inn_key'],
    how='inner',
)
print(
    'keys with n_agr =', len(keys_nagr),
    '| coverage agr keys =',
    round(
        keys_nagr[['report_month', 'agr_id_key', 'inn_key']].drop_duplicates().shape[0]
        / max(len(keys_7d), 1),
        4,
    ),
)

term_parts = []
for month_label, g_month in keys_nagr.groupby('report_month'):
    month_start = f'{month_label}-01'
    month_end = (pd.Timestamp(month_start) + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    n_agrs = clean_keys(g_month['n_agr'].tolist())
    print(f'[{month_label}] n_agr in perimeter =', len(n_agrs))
    for chunk in iter_chunks(n_agrs, NAGR_CHUNK):
        nagr_in = sql_in(chunk)
        sql_terms = f"""
        select distinct
          cast('{month_label}' as string) as report_month,
          cast('{month_start}' as string) as snapshot_month_start,
          cast(t.n_agr as string) as n_agr,
          cast(a.n_cmp_client as string) as n_cmp_client,
          cast(t.c_nmrc as string) as c_nmrc,
          cast(p.c_nter as string) as c_nter,
          cast(p.c_pos_serial as string) as c_pos_serial,
          cast(p.d_ter_install as date) as d_ter_install,
          cast(p.d_ter_close as date) as d_ter_close
        from ods_alpha.scd1_agr_terms t
        join ods_alpha.scd1_agreements a
          on cast(a.n_agr as string) = cast(t.n_agr as string)
        join ods_alpha.scd1_merchants m
          on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
        join ods_alpha.scd1_pos_terminals p
          on cast(p.c_nmrc as string) = cast(t.c_nmrc as string)
        where cast(t.n_agr as string) in ({nagr_in})
          and t.c_nmrc is not null
          and p.c_nter is not null
          and p.c_pos_serial is not null
          and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
          and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
          and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
          and cast(p.d_ter_install as date) is not null
          and cast(p.d_ter_install as date) <= cast('{month_end}' as date)
          and coalesce(cast(p.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
          and coalesce(t.ods_deleted_flg, '0') <> '1'
          and coalesce(m.ods_deleted_flg, '0') <> '1'
          and coalesce(p.ods_deleted_flg, '0') <> '1'
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and upper(trim(cast(a.acq_class as string))) = 'SA'
        """
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            part = imp.fetch(sql_terms)
        if part is not None and len(part):
            term_parts.append(part)

if not term_parts:
    raise RuntimeError('No terminals found for final_df perimeter — check agr_terms / month overlap')

term_perim_7d = pd.concat(term_parts, ignore_index=True)
for c in ['report_month', 'n_agr', 'n_cmp_client', 'c_nter', 'c_pos_serial']:
    term_perim_7d[c] = term_perim_7d[c].astype(str).str.strip()
term_perim_7d['d_ter_install'] = pd.to_datetime(term_perim_7d['d_ter_install'], errors='coerce')
term_perim_7d['d_ter_close'] = pd.to_datetime(term_perim_7d['d_ter_close'], errors='coerce')
_allowed = keys_nagr[['report_month', 'n_agr']].drop_duplicates()
term_perim_7d = term_perim_7d.merge(_allowed, on=['report_month', 'n_agr'], how='inner')
print(
    'term_perim rows =', len(term_perim_7d),
    '| serials =', term_perim_7d['c_pos_serial'].nunique(),
    '| c_nter =', term_perim_7d['c_nter'].nunique(),
)


def _client_col(df):
    cmp_n = df['n_cmp_client'].replace({'None': np.nan, 'nan': np.nan, '': np.nan})
    if cmp_n.notna().any():
        return 'n_cmp_client'
    return 'n_agr'


client_col = _client_col(term_perim_7d)
print('client grain for handover =', client_col)

serial_month = (
    term_perim_7d.groupby(['report_month', 'c_pos_serial'], as_index=False)
    .agg(
        n_c_nter=('c_nter', 'nunique'),
        n_client=(client_col, 'nunique'),
        n_agr=('n_agr', 'nunique'),
        c_nter_list=('c_nter', lambda s: '|'.join(sorted(set(s.astype(str))))),
        client_list=(client_col, lambda s: '|'.join(sorted(set(s.astype(str))))),
        n_agr_list=('n_agr', lambda s: '|'.join(sorted(set(s.astype(str))))),
    )
)


def _is_close_then_install(g, month_label):
    ms = pd.Timestamp(f'{month_label}-01')
    me = ms + pd.offsets.MonthEnd(0)
    facts = (
        g.groupby('c_nter', as_index=False)
        .agg(
            d_ter_install=('d_ter_install', 'min'),
            d_ter_close=('d_ter_close', 'max'),
        )
    )
    if len(facts) < 2:
        return False
    closed_in = facts['d_ter_close'].notna() & (facts['d_ter_close'] >= ms) & (facts['d_ter_close'] <= me)
    installed_in = facts['d_ter_install'].notna() & (facts['d_ter_install'] >= ms) & (facts['d_ter_install'] <= me)
    closers = set(facts.loc[closed_in, 'c_nter'])
    openers = set(facts.loc[installed_in, 'c_nter'])
    if not closers or not openers:
        return False
    # classic handoff: different c_nter closed vs installed in the same month
    return bool(closers - openers) or bool(openers - closers) or (len(closers) >= 2)


cti_flags = []
for (rm, serial), g in term_perim_7d.groupby(['report_month', 'c_pos_serial']):
    cti_flags.append({
        'report_month': rm,
        'c_pos_serial': serial,
        'close_then_install': _is_close_then_install(g, rm),
    })
cti_df = pd.DataFrame(cti_flags)
serial_month = serial_month.merge(cti_df, on=['report_month', 'c_pos_serial'], how='left')
serial_month['close_then_install'] = serial_month['close_then_install'].fillna(False)
serial_month['handover_same_month'] = (serial_month['n_c_nter'] >= 2) & (serial_month['n_client'] >= 2)
serial_month['extra_c_nter_rows'] = np.where(
    serial_month['handover_same_month'],
    serial_month['n_c_nter'] - 1,
    0,
)

handover_stats = (
    serial_month.groupby('report_month', as_index=False)
    .agg(
        serials_in_perimeter=('c_pos_serial', 'nunique'),
        serials_multi_c_nter=('handover_same_month', 'sum'),
        serials_close_then_install=('close_then_install', 'sum'),
        extra_c_nter_rows=('extra_c_nter_rows', 'sum'),
    )
)
handover_stats['share_handover'] = (
    handover_stats['serials_multi_c_nter'] / handover_stats['serials_in_perimeter']
).round(4)
handover_stats['share_close_then_install'] = (
    handover_stats['serials_close_then_install'] / handover_stats['serials_in_perimeter']
).round(4)

print('=== Mid-month serial handover by month (final_df perimeter) ===')
display(handover_stats)

period_row = {
    'serials_in_perimeter': int(serial_month['c_pos_serial'].nunique()),
    'serial_month_rows': int(len(serial_month)),
    'serials_multi_c_nter': int(serial_month['handover_same_month'].sum()),
    'serials_close_then_install': int(serial_month['close_then_install'].sum()),
    'extra_c_nter_rows': int(serial_month['extra_c_nter_rows'].sum()),
}
period_row['share_handover'] = round(
    period_row['serials_multi_c_nter'] / max(period_row['serial_month_rows'], 1), 4
)
print('=== Period totals (serial×month grain for shares) ===')
display(pd.DataFrame([period_row]))

sample_7d = (
    serial_month.loc[serial_month['handover_same_month']]
    .sort_values(['report_month', 'n_c_nter', 'c_pos_serial'], ascending=[True, False, True])
    .head(20)
)
print('=== Sample handover serials (up to 20) ===')
display(sample_7d[[
    'report_month', 'c_pos_serial', 'n_c_nter', 'n_client', 'n_agr',
    'c_nter_list', 'client_list', 'n_agr_list', 'close_then_install', 'extra_c_nter_rows',
]])

# multi-agr within same perimeter (one c_nter → many n_agr) for scale compare
nter_agr = (
    term_perim_7d.groupby(['report_month', 'c_nter'], as_index=False)
    .agg(n_agr=('n_agr', 'nunique'))
)
multi_agr_perim = int((nter_agr['n_agr'] >= 2).sum())
multi_agr_nters = int(nter_agr.loc[nter_agr['n_agr'] >= 2, 'c_nter'].nunique())
extra_agr_rows = int((nter_agr['n_agr'] - 1).clip(lower=0).sum())

multi_agr_7c = None
if 'multi_agr_df' in globals() and multi_agr_df is not None and len(multi_agr_df):
    try:
        multi_agr_7c = int(pd.to_numeric(multi_agr_df.iloc[0].get('terminals_multi_agr'), errors='coerce') or 0)
    except Exception:
        multi_agr_7c = None

print('=== Scale vs multi-agr ===')
print(f'handover serial×month rows: {period_row["serials_multi_c_nter"]}')
print(f'extra_c_nter_rows (est. double-count if grain=c_nter): {period_row["extra_c_nter_rows"]}')
print(
    f'multi-agr (c_nter×month with ≥2 n_agr) in same perimeter: '
    f'{multi_agr_perim} rows / {multi_agr_nters} distinct c_nter'
)
print(f'extra_agr_rows (est. double-count if SUM by agr): {extra_agr_rows}')
if multi_agr_7c is not None:
    print(f'7c multi-agr terminals (amort>0 @ sample month): {multi_agr_7c}')

share = period_row['share_handover']
extra_h = period_row['extra_c_nter_rows']
if share < 0.005 and extra_h < 100:
    verdict = (
        'РЕДКИЙ ШУМ: mid-month serial transfer почти не влияет на ×5 vs Excel; '
        'главный риск — multi-agr.'
    )
elif extra_h < max(extra_agr_rows * 0.15, 1):
    verdict = (
        f'ЗАМЕТНО МЕНЬШЕ multi-agr: extra_c_nter_rows={extra_h} << extra_agr_rows={extra_agr_rows}; '
        'handover не главный драйвер раздувания amort.'
    )
elif extra_h >= extra_agr_rows * 0.5:
    verdict = (
        f'СОПОСТАВИМО с multi-agr: extra_c_nter_rows={extra_h} vs extra_agr_rows={extra_agr_rows}; '
        'нужен dedupe по serial в месяце.'
    )
else:
    verdict = (
        f'ВТОРОСТЕПЕННЫЙ ВКЛАД: handover share={share}, extra_c_nter_rows={extra_h}; '
        f'multi-agr extra_agr_rows={extra_agr_rows} всё ещё крупнее.'
    )
print('VERDICT:', verdict)

qc_handover_csv = _qc_out_dir / 'amortization_serial_midmonth_handover_by_month.csv'
qc_sample_csv = _qc_out_dir / 'amortization_serial_midmonth_handover_sample.csv'
handover_stats.to_csv(qc_handover_csv, index=False, encoding='utf-8-sig')
sample_7d.to_csv(qc_sample_csv, index=False, encoding='utf-8-sig')
print('Saved:', qc_handover_csv)
print('Saved:', qc_sample_csv)
